In [1]:
# =============================================================================
# Static Layer Pipeline — Ontario (confirmed working configuration)
#
# Confirmed scales from test exports:
#   elevation    — setDefaultProjection scale=300  ✅
#   population   — setDefaultProjection scale=300  ✅
#   land_cover   — setDefaultProjection scale=1000 ✅ (300 OOM, 1000 works)
#   road_proximity — setDefaultProjection scale=300 ✅
# =============================================================================

import ee
import time

KEY_FILE        = 'ru-thesis-2026-66238f2c1197.json'
SERVICE_ACCOUNT = 'ru-thesis-gee-api@ru-thesis-2026.iam.gserviceaccount.com'
PROJECT         = 'ru-thesis-2026'

credentials = ee.ServiceAccountCredentials(SERVICE_ACCOUNT, KEY_FILE)
ee.Initialize(credentials, project=PROJECT)

# ── Grid & mask ───────────────────────────────────────────────────────────────
era5_sample = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR").first()
GRID_PROJ   = era5_sample.select(0).projection()
GRID_SCALE  = GRID_PROJ.nominalScale()

ontario_fc   = (ee.FeatureCollection("FAO/GAUL/2015/level1")
                .filter(ee.Filter.eq('ADM0_NAME', 'Canada'))
                .filter(ee.Filter.eq('ADM1_NAME', 'Ontario')))
ontario_geom = ontario_fc.geometry().simplify(4500)

PROVINCE_RASTER = (ontario_fc
                   .map(lambda f: f.set('province_code', 9))
                   .reduceToImage(['province_code'], ee.Reducer.first())
                   .reproject(crs=GRID_PROJ)
                   .rename('province_id')
                   .toByte())

era5_coverage = (ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
                 .filterDate('2024-07-01', '2024-07-02')
                 .first()
                 .select('temperature_2m')
                 .mask()
                 .reproject(crs=GRID_PROJ))

ONTARIO_MASK    = (PROVINCE_RASTER.eq(9).And(era5_coverage).selfMask())
PROVINCE_RASTER = PROVINCE_RASTER.updateMask(ONTARIO_MASK)

INTERMEDIATE_SCALE = 300

# ── Elevation & slope (confirmed: scale=300) ──────────────────────────────────
cdem_raw = (ee.ImageCollection("NRCan/CDEM")
            .filterBounds(ontario_geom)
            .select('elevation')
            .mean()
            .clip(ontario_geom))

elevation_300m = (cdem_raw
                  .setDefaultProjection(crs='EPSG:4269', scale=300)
                  .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=65535)
                  .reproject(crs=GRID_PROJ, scale=INTERMEDIATE_SCALE))

slope_300m = ee.Terrain.slope(elevation_300m)

elevation = (elevation_300m
             .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=65535)
             .reproject(crs=GRID_PROJ)
             .rename('elevation_m'))

slope = (slope_300m
         .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=65535)
         .reproject(crs=GRID_PROJ)
         .rename('slope_deg'))

# ── Land cover (confirmed: scale=1000) ────────────────────────────────────────
LAND_COVER = (ee.Image("USGS/NLCD_RELEASES/2020_REL/NALCMS")
              .select('landcover')
              .clip(ontario_geom)
              .setDefaultProjection(crs='EPSG:4326', scale=1000)
              .reduceResolution(reducer=ee.Reducer.mode(), maxPixels=65535)
              .reproject(crs=GRID_PROJ, scale=INTERMEDIATE_SCALE)
              .reduceResolution(reducer=ee.Reducer.mode(), maxPixels=65535)
              .reproject(crs=GRID_PROJ)
              .rename('land_cover')
              .toByte())

# ── WorldPop population density (confirmed: scale=300) ────────────────────────
def get_pop_density(pop_year):
    return (ee.ImageCollection("WorldPop/GP/100m/pop")
            .filter(ee.Filter.eq('country', 'CAN'))
            .filter(ee.Filter.calendarRange(pop_year, pop_year, 'year'))
            .filterBounds(ontario_geom)
            .first()
            .select('population')
            .clip(ontario_geom)
            .setDefaultProjection(crs='EPSG:4326', scale=300)
            .reduceResolution(reducer=ee.Reducer.sum(), maxPixels=65535)
            .reproject(crs=GRID_PROJ, scale=INTERMEDIATE_SCALE)
            .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=65535)
            .reproject(crs=GRID_PROJ)
            .rename('pop_density'))

# ── Road proximity (confirmed: scale=300) ─────────────────────────────────────
roads_ontario = (ee.FeatureCollection(
                     "projects/ru-thesis-2026/assets/road_network_2024")
                 .filterBounds(ontario_geom))

dist_raw = roads_ontario.distance(searchRadius=100000)

dist_300m = (dist_raw
             .setDefaultProjection(crs='EPSG:4326', scale=300)
             .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=65535)
             .reproject(crs=GRID_PROJ, scale=INTERMEDIATE_SCALE))

road_proximity = (dist_300m
                  .reduceResolution(reducer=ee.Reducer.mean(), maxPixels=65535)
                  .reproject(crs=GRID_PROJ)
                  .rename('road_proximity_m'))

# ── Version assignment ────────────────────────────────────────────────────────
def get_worldpop_year(study_year):
    return min(study_year, 2020)

# ── Asset folder ──────────────────────────────────────────────────────────────
STATIC_FOLDER = f"projects/{PROJECT}/assets/Ontario_Static_2010-2024"

try:
    ee.data.createAsset({'type': 'ImageCollection'}, STATIC_FOLDER)
    print(f"Created: {STATIC_FOLDER}")
except ee.EEException:
    print(f"Already exists: {STATIC_FOLDER}")

# ── Single-year test export before full run ───────────────────────────────────
# TEST_YEAR   = 2024
#pop_year    = get_worldpop_year(TEST_YEAR)
# pop_density = get_pop_density(pop_year)

#static_stack = (elevation
#                .addBands([slope,
#                           LAND_COVER,
#                           pop_density,
#                           road_proximity,
#                           PROVINCE_RASTER])
#                .updateMask(ONTARIO_MASK)
#                .set({
#                    'study_year'        : TEST_YEAR,
#                    'nalcms_edition'    : 2020,
#                    'worldpop_year'     : pop_year,
#                    'elevation_source'  : 'NRCan/CDEM (2011)',
#                    'land_cover_source' : 'USGS NALCMS 2020',
#                    'road_source'       : 'Canada road network 2024',
#                    'system:time_start' : ee.Date(f'{TEST_YEAR}-01-01').millis()
#                }))

# print(f"Bands: {static_stack.bandNames().getInfo()}")

# task = ee.batch.Export.image.toAsset(
#     image       = static_stack,
#    description = f'Static_{TEST_YEAR}_test',
#     assetId     = f'{STATIC_FOLDER}/Static_{TEST_YEAR}_test',
#     region      = ontario_geom,
#    crs         = GRID_PROJ,
#    scale       = GRID_SCALE,
#    maxPixels   = 1e13
#)
# task.start()
# print(f"Combined test export submitted for {TEST_YEAR}.")
# print(f"Monitor: https://code.earthengine.google.com/tasks")



Already exists: projects/ru-thesis-2026/assets/Ontario_Static_2010-2024


In [ ]:
# ── Full export loop ────────────────────────────
STUDY_YEARS = list(range(2010, 2025))
BATCH_SIZE  = 5    # static exports are heavy — submit in small batches
SLEEP_SEC   = 30

print(f"\nSubmitting {len(STUDY_YEARS)} static export tasks...")
print(f"{'Year':<8} {'WorldPop':<12} Status")
print("-" * 35)

for i, study_year in enumerate(STUDY_YEARS):
     pop_year    = get_worldpop_year(study_year)
     pop_density = get_pop_density(pop_year)

     static_stack = (elevation
                     .addBands([slope,
                                LAND_COVER,
                                pop_density,
                                road_proximity,
                                PROVINCE_RASTER])
                     .updateMask(ONTARIO_MASK)
                     .set({
                         'study_year'        : study_year,
                         'nalcms_edition'    : 2020,
                         'worldpop_year'     : pop_year,
                         'elevation_source'  : 'NRCan/CDEM (2011)',
                         'land_cover_source' : 'USGS NALCMS 2020',
                         'road_source'       : 'Canada road network 2024',
                         'system:time_start' : ee.Date(f'{study_year}-01-01').millis()
                     }))

     task = ee.batch.Export.image.toAsset(
         image       = static_stack,
         description = f'Static_{study_year}',
         assetId     = f'{STATIC_FOLDER}/Static_{study_year}',
         region      = ontario_geom,
         crs         = GRID_PROJ,
         scale       = GRID_SCALE,
         maxPixels   = 1e13
     )
     task.start()
     print(f"  {study_year:<8} {pop_year:<12} submitted")

     if (i + 1) % BATCH_SIZE == 0:
         print(f"  Pausing {SLEEP_SEC}s...")
         time.sleep(SLEEP_SEC)

print("All static tasks submitted.")
print("Monitor: https://code.earthengine.google.com/tasks")